In [1]:
import pandas as pd

# Read the orders CSV file into a DataFrame
# parse_dates converts the 'order_date' column from text to datetime format
orders = pd.read_csv("orders.csv", parse_dates=["order_date"])

# Select only the required columns from the orders DataFrame
cols = orders[["order_id", "category", "price", "quantity", "city"]]

# Display the first 3 rows of the selected columns
cols.head(3)

,order_id,category,price,quantity,city
0,ORD10001,Electronics,1299.0,1,Mumbai
1,ORD10002,Fashion,2199.0,1,Mumbai
2,ORD10003,Home & Kitchen,649.0,2,Mumbai


In [2]:
# Calculate revenue for each order line
# Revenue = Price × Quantity
# Pandas performs this calculation for the entire column at once
orders["revenue"] = orders["price"] * orders["quantity"]

# Display the selected columns along with the newly created revenue column
orders[["order_id", "price", "quantity", "revenue"]].head(3)

,order_id,price,quantity,revenue
0,ORD10001,1299.0,1,1299.0
1,ORD10002,2199.0,1,2199.0
2,ORD10003,649.0,2,1298.0


In [4]:
# Create a boolean mask to identify:
# 1. Electronics orders
# 2. Orders from Mumbai
# 3. Orders with revenue greater than ₹2,000
# & (and) / | (or) 
# Splitting a single line lengthy code to multi-lines for easy understanding
mask =  (orders["category"] == "Electronics") & \
        (orders["city"] == "Mumbai") & \
        (orders["revenue"] > 2000)

# Apply the filter and display selected columns
orders[mask][["order_id", "brand", "revenue"]].head()

,order_id,brand,revenue
3,ORD10004,Samsung,18999.0
12,ORD10009,Samsung,2013.0
44,ORD10029,Realme,5403.0
152,ORD10095,OnePlus,2160.0
192,ORD10120,OnePlus,4071.0


In [5]:
# Select orders from Mumbai, Delhi, or Bengaluru
# .isin() is useful when checking a column against multiple values
metro = orders[
    orders["city"].isin(["Mumbai", "Delhi", "Bengaluru"])
]

# Count the number of matching order line items
print(len(metro), "line items in the three biggest metros")

7577 line items in the three biggest metros


In [9]:
# groupby(col)[measure].agg(...)
#Group orders by category
# Sum the revenue for each category
# Sort categories from highest to lowest revenue
by_category = (
    orders.groupby("category")["revenue"]
          .sum()
          .sort_values(ascending=False)
)

# Display total revenue by category
by_category

category
Electronics       9663403.0
Home & Kitchen    3496498.0
Fashion           3133305.0
Sports            3034279.0
Beauty            1129002.0
Books              771678.0
Name: revenue, dtype: float64

In [10]:
# Group the orders by category and calculate multiple metrics
summary = orders.groupby("category").agg(
    # Total revenue generated by the category
    total_revenue=("revenue", "sum"),
    # Number of order line items (rows)
    line_items=("order_id", "count"),
    # Number of unique/distinct orders
    distinct_orders=("order_id", "nunique"),
    # Average product price in the category
    avg_price=("price", "mean"),
)

# Sort categories by total revenue from highest to lowest
summary.sort_values("total_revenue", ascending=False)

,total_revenue,line_items,distinct_orders,avg_price
category,,,,
Electronics,9663403.0,4068,3418,1861.284415
Home & Kitchen,3496498.0,2137,1942,1249.909686
Fashion,3133305.0,2429,2202,905.526554
Sports,3034279.0,1591,1504,1558.891892
Beauty,1129002.0,1480,1385,368.279054
Books,771678.0,775,750,565.285161


In [11]:
# Group orders by city
# Calculate total revenue for each city
# Sort from highest to lowest revenue
# Select the top 3 cities
top_cities = (
    orders.groupby("city")["revenue"]
          .sum()
          .sort_values(ascending=False)
          .head(3)
)

# Display the top 3 cities by revenue
top_cities

city
Mumbai       5223311.0
Delhi        4037024.0
Bengaluru    3715412.0
Name: revenue, dtype: float64

In [12]:
# Load the customers data
# Convert signup_date into datetime format
customers = pd.read_csv("customers.csv", parse_dates=["signup_date"])

# Load the products data
products = pd.read_csv("products.csv")

# Merge orders with customer information
# customer_id is the common key between the two tables
# left join keeps every order even if customer information is missing
df = orders.merge(
    customers,
    on="customer_id",
    how="left",
    suffixes=("", "_cust")
)

# Merge the result with product information
# product_id is the common key between orders and products
df = df.merge(
    products,
    on="product_id",
    how="left",
    suffixes=("", "_prod")
)

# Display important columns from the final joined DataFrame
df[
    [
        "order_id",
        "customer_id",
        "segment",
        "category",
        "rating",
        "revenue"
    ]
].head(3)

,order_id,customer_id,segment,category,rating,revenue
0,ORD10001,CUST042,Premium,Electronics,4.2,1299.0
1,ORD10002,CUST119,Regular,Fashion,4.2,2199.0
2,ORD10003,CUST042,Premium,Home & Kitchen,4.4,1298.0


In [13]:
# Group orders by customer segment
# Calculate total revenue, average revenue, and number of rows
segment_summary = (
    df.groupby("segment")["revenue"]
      .agg(["sum", "mean", "count"])
)

# Display the results
segment_summary

,sum,mean,count
segment,,,
Premium,10630927.0,2223.113133,4782
Regular,10295572.0,1378.255957,7470


In [14]:
# Count missing values in every column
# The lambda function keeps only columns with missing values
missing_values = df.isna().sum()[lambda s: s > 0]

# Display columns that contain missing values
missing_values

brand      120
segment    228
dtype: int64

In [15]:
# Replace missing brand values with "Unknown"
# This keeps the rows while clearly identifying missing brand information
df["brand"] = df["brand"].fillna("Unknown")

# Find the most frequently occurring customer segment
# mode()[0] returns the first most-common value
most_common_segment = df["segment"].mode()[0]

# Replace missing segment values with the most common segment
df["segment"] = df["segment"].fillna(most_common_segment)

# Check the total number of missing values remaining
df.isna().sum().sum()

np.int64(0)

In [16]:
# Convert category from object/string type to pandas category type
# This can reduce memory usage for repeated categorical values
df["category"] = df["category"].astype("category")

# Convert segment to categorical data type
df["segment"] = df["segment"].astype("category")

# Ensure order_date is stored as a proper datetime
df["order_date"] = pd.to_datetime(df["order_date"])